# core

> Async Documenso v2 API client built on fastspec

In [ ]:
#| default_exp core

Create a client with `documenso_client()` and call generated operations:
`await cli.envelope.envelope_create(...)`, `await cli.document.document_get(...)`, etc.

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import expect_fail

In [ ]:
#| export
import os, json, httpx
from dataclasses import replace
from fastcore.utils import *
from fastcore.script import call_parse
from fastspec.spec import SpecParser
from fastspec.oapi import OpenAPIClient
from nbdev.config import get_config

from fastdocumenso._spec import spec as _spec

Documenso's multipart endpoints are unusual: dict and list parts such as `payload` must arrive as JSON strings rather than as repeated form fields. `_encode_form` is passed to the client as its form encoder so callers can keep writing plain Python values.

In [ ]:
#| export
def _encode_form(d):
    "Documenso multipart endpoints expect dict/list parts (e.g. `payload`) as JSON strings"
    return {k: json.dumps(v) if isinstance(v,(dict,list)) else v for k,v in d.items()}

In [ ]:
_encode_form({'payload': {'title': 'My NDA'}, 'name': 'doc.pdf'})

{'payload': '{"title": "My NDA"}', 'name': 'doc.pdf'}

`documenso_client` reads the token from `$DOCUMENSO_API_KEY` and the base URL from `$DOCUMENSO_API_URL`, falling back to the URL recorded in the spec. Operations are grouped as attributes, and each operation is an awaitable method.

In [ ]:
#| export
def documenso_client(
    api_key:str|None=None,  # Documenso API token; defaults to $DOCUMENSO_API_KEY
    base_url:str|None=None, # defaults to $DOCUMENSO_API_URL or the documenso.com cloud
)->OpenAPIClient:           # groups as attributes, ops as awaitable methods
    "Async Documenso v2 API client"
    spec = SpecParser.from_dict(_spec)
    spec.base_url = base_url or os.environ.get('DOCUMENSO_API_URL', spec.base_url)
    return OpenAPIClient(spec, headers={'Authorization': api_key or os.environ['DOCUMENSO_API_KEY']}, form_encoder=_encode_form)

In [ ]:
#| notest
cli = documenso_client()
list(cli.groups), len(cli.ops)

(['envelope', 'document', 'template', 'folder', 'embedding'], 89)

```python
d = await cli.document.document_get(document_id=1819393)
d['title']
```
```
'Test Document'
```

## Updating the spec

`_spec.py` is a snapshot of the Documenso OpenAPI spec. Regenerate it with the `fastdocumenso_update` command when the API changes.

In [ ]:
#| export
SPEC_URL = 'https://app.documenso.com/api/v2/openapi.json'

Documenso's OpenAPI spec marks file-upload parameters with an empty item schema (`{}`) instead of the standard `{'type': 'string', 'format': 'binary'}`, so `SpecParser.from_openapi` doesn't detect them as file params. Without that detection, `files` is treated as an ordinary JSON field — and passing a real file object then fails, because `httpx`/`json.dumps` can't serialize a file object.

In [ ]:
#| notest
raw = httpx.get(SPEC_URL, follow_redirects=True).raise_for_status().json()
raw['paths']['/document/create']['post']['requestBody']['content']['multipart/form-data']['schema']['properties']['file']

{}

Building a client straight from that raw spec shows the failure. `files` is treated as an ordinary JSON field, so a file object reaches `json.dumps` and raises:

In [ ]:
#| notest
parser_unpatched = SpecParser.from_openapi(dict2obj(raw))
cli_unpatched = OpenAPIClient(parser_unpatched, headers={'Authorization': os.environ['DOCUMENSO_API_KEY']}, form_encoder=_encode_form)

Path('dummy.pdf').write_bytes(b'%PDF-1.4\n%dummy pdf for testing\n')
with expect_fail(TypeError, contains='JSON serializable'):
    await cli_unpatched.envelope.envelope_create(payload={'title': 't', 'type': 'DOCUMENT'}, files=[open('dummy.pdf', 'rb')])

`_FILE_OPS` maps each affected operation to its file parameter name(s), and `update_spec` `replace`s those ops' `file_params` after parsing to patch this in manually — that's what lets `documenso_client()` correctly route `files` as a multipart upload instead of a JSON field.

In [ ]:
#| export
_FILE_OPS = {'envelope_item_create_many': 'files', 'envelope_create': 'files', 'envelope_use': 'files',
             'document_create': 'file', 'template_create_template': 'file'}

In [ ]:
#| export
@call_parse
def update_spec(
    url:str=SPEC_URL, # OpenAPI spec URL to fetch
    out:str=None, # Output path; defaults to `_spec.py` in the package
):
    "Regenerate `_spec.py` from the Documenso OpenAPI spec"
    out = Path(out) if out else get_config().lib_path/'_spec.py'
    raw = httpx.get(url, follow_redirects=True).raise_for_status().json()
    parser = SpecParser.from_openapi(dict2obj(raw))
    parser.ops = [replace(o, file_params=[_FILE_OPS[o.name]]) if o.name in _FILE_OPS else o for o in parser.ops]
    parser.save(out)
    return f"Saved {len(parser.ops)} ops to {out}"

## Export -

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()